In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Data Mining Spark Data Lake Assignment") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
import os

os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake structure created.")

Data Lake structure created.


In [4]:
bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

df.show(5)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|       680|               5|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|      3080|              12|           UPI|      Pune|   Maharashtra|2025-05-06|     Shipped|
|  100004|      C0051|

In [5]:
print("Schema:")
df.printSchema()

print("First 5 rows:")
df.show(5, truncate=False)

print("Total records:", df.count())

Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First 5 rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+---

In [6]:
print("Schema:")
df.printSchema()

print("First 5 rows:")
df.show(5, truncate=False)

print("Total records:", df.count())

Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First 5 rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+---

In [7]:
print("Columns:")
print(df.columns)

Columns:
['order_id', 'customer_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'payment_method', 'city', 'state', 'order_date', 'order_status']


In [8]:
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", total_rows - distinct_rows)

Total rows: 1000
Distinct rows: 990
Duplicate rows: 10


In [9]:
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [10]:
for c in ["customer_name", "product", "category",
          "payment_method", "city", "state", "order_status"]:
    print(f"\n--- {c} ---")
    df.select(c).distinct().show(50, truncate=False)


--- customer_name ---
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai      |
|Pooja Jain        |
|Siddharth Kapoor  |
|Keerthi Tripathi  |
|Aditya Chopra     |
|Arjun Agarwal     |
|Vivaan Nair       |
|Deepak Pandey     |
|Pooja Verma       |
|Neha Sharma       |
|Divya Chopra      |
|Aarav Nair        |
|Kavya Malhotra    |
|Sanjay Shetty     |
|Vivaan Pandey     |
|Aditi Patel       |
|Nandini Das       |
|Aditi Tripathi    |
|Siddharth Nair    |
|Divya Sharma      |
|Ananya Verma      |
|Ananya Mishra     |
|Meera Reddy       |
|Varun Agarwal     |
|Pranav Agarwal    |
|Siddharth Iyer    |
|Kavya Shar

In [11]:
silver_df = df.dropDuplicates()

In [21]:
string_columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_date",
    "order_status"
]

for c in string_columns:
    silver_df = silver_df.withColumn(
        c,
        trim(col(c))
    )


In [12]:
silver_df = silver_df.withColumn(
    "category",
    initcap(col("category"))
)
silver_df = silver_df.withColumn(
    "city",
    initcap(col("city"))
)


In [13]:
silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [14]:
silver_df = silver_df.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

In [15]:
print("Invalid quantity:")
silver_df.filter(col("quantity") <= 0).show()

print("Invalid unit price:")
silver_df.filter(col("unit_price") < 0).show()

print("Invalid discount:")
silver_df.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).show()

Invalid quantity:
+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|             product|       category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|
+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|  100375|      C0027|   Harsh Jain|    Basmati Rice 5kg|        Grocery|       0|       460|              10|      Debit Card|    Mumbai|   Maharashtra|2025-05-03|     Shipped|
|  100919|      C0214|Neha Tripathi|              Jacket|       Clothing|      -2|      3600|              20|             UPI|    Jaipur|     Rajasthan|2025-02-06|   Delivered|
|  100926|      C0002|  Deepak Jain|         Ceiling Fan|Home Appliances|      -2|      3070

In [16]:
silver_df = silver_df.filter(col("quantity") > 0)

silver_df = silver_df.filter(col("unit_price") >= 0)

silver_df = silver_df.filter(
    (col("discount_percent") >= 0) &
    (col("discount_percent") <= 100)
)

In [17]:
silver_df.select("order_date").distinct().show(50, truncate=False)

+----------+
|order_date|
+----------+
|2026-02-01|
|2025-06-03|
|2025-02-25|
|2026-02-27|
|2025-08-17|
|2026-07-23|
|2026-07-04|
|2025-09-14|
|2025-03-20|
|2025-04-05|
|2026-04-15|
|2026-06-05|
|2025-01-03|
|2026-03-15|
|2025-12-30|
|2025-08-18|
|2026-09-11|
|2025-06-16|
|2026-09-06|
|2025-07-15|
|2026-08-24|
|2025/07/14|
|2025-05-06|
|2025-04-12|
|2026-02-15|
|2025-07-21|
|2025-05-23|
|2025-10-09|
|2026-01-05|
|2025-01-23|
|2026-01-28|
|2025-06-14|
|2026-02-25|
|2025-07-16|
|2025-01-28|
|2026-01-09|
|2025-03-12|
|2025-03-26|
|2025-12-14|
|2025-07-06|
|2025-07-08|
|2026-01-02|
|2025-05-09|
|2025-03-07|
|2025-06-01|
|2025-06-30|
|2025-07-29|
|2025-01-22|
|2026-01-13|
|2026-01-08|
+----------+
only showing top 50 rows


In [18]:
silver_df = silver_df.withColumn(
    "order_date",
    regexp_replace(col("order_date"), "/", "-")
)
silver_df = silver_df.withColumn(
    "order_date",
    try_to_timestamp(
        col("order_date"),
        lit("yyyy-MM-dd")
    ).cast("date")
)


In [19]:
invalid_dates = silver_df.filter(
    col("order_date").isNull()
).count()

print("Invalid date records:", invalid_dates)

silver_df = silver_df.filter(
    col("order_date").isNotNull()
)

Invalid date records: 5


In [23]:
print("Silver dataframe created successfully.")
print("Silver row count:", silver_df.count())

silver_df.printSchema()
silver_df.show(10, truncate=False)

Silver dataframe created successfully.
Silver row count: 961
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+----

In [24]:
before_count = df.count()
after_count = silver_df.count()

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing:", after_count)
print("Rows removed:", before_count - after_count)

Rows before preprocessing: 1000
Rows after preprocessing: 961
Rows removed: 39


In [26]:
silver_output = "/content/data_lake/silver/preprocessed_sales"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_output)

print("Silver data saved successfully.")

Silver data saved successfully.


In [27]:
import glob
import shutil

csv_files = glob.glob(
    "/content/data_lake/silver/preprocessed_sales/part-*.csv"
)

silver_csv = "/content/data_lake/silver/sales_silver.csv"

shutil.copy(csv_files[0], silver_csv)

print("Created:")
print(silver_csv)

Created:
/content/data_lake/silver/sales_silver.csv


In [29]:
print("========================================")
print("       FINAL DATA LAKE VERIFICATION")
print("========================================")

print("Bronze row count :", df.count())
print("Silver row count :", silver_check.count())

print("\nSilver Schema:")
silver_check.printSchema()

print("\nSample Silver Records:")
silver_check.show(5, truncate=False)

print("\nRemaining Null Values:")
silver_check.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_check.columns
]).show()

       FINAL DATA LAKE VERIFICATION
Bronze row count : 1000
Silver row count : 961

Silver Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


Sample Silver Records:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |

## Preprocessing Summary

The raw sales.csv file was preserved unchanged in the Bronze layer.

The following preprocessing operations were performed using Apache Spark:

1. Removed exact duplicate records.
2. Checked missing values and replaced missing categorical values with "Unknown".
3. Trimmed extra spaces from text fields.
4. Standardized capitalization of applicable text fields.
5. Removed records with zero or negative quantities.
6. Removed records with negative unit prices.
7. Removed records with discount percentages outside the 0–100 range.
8. Standardized inconsistent date separators.
9. Converted order_date into a date data type.
10. Removed clearly invalid dates that could not be parsed.
11. Saved the processed data in the Silver layer.
12. Read the Silver dataset back using Spark for verification.